<a href="https://colab.research.google.com/github/Amryasser456/Amryasser456/blob/main/test002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers==0.0.34 trl==0.12.0 peft accelerate bitsandbytes
!pip install unsloth_zoo

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-pw5p053d/unsloth_150191d288134a1faf161bad97b3b76a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-pw5p053d/unsloth_150191d288134a1faf161bad97b3b76a
  Resolved https://github.com/unslothai/unsloth.git to commit 01f2e289a7376a3a4d71a2e39bed025a72df0273
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.8 MB/s eta 0:00:0

In [ ]:
# =========================
# 1️⃣ Imports
# =========================
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch

# =========================
# 2️⃣ Config
# =========================
max_seq_length = 1024
load_in_4bit = True
model_name = "stabilityai/ar-stablelm-2-chat"   # Change if needed

# =========================
# 3️⃣ Load Model (Unsloth)
# =========================
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

# =========================
# 4️⃣ Add LoRA
# =========================
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# =========================
# 5️⃣ Load Dataset
# =========================
dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/input.json",
)

dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# =========================
# 6️⃣ Format Prompts (StableLM Chat Style)
# =========================
def format_prompts(examples):
    texts = []
    for inst, bef, aft in zip(
        examples["prompt"],
        examples["before"],
        examples["after"]
    ):
        text = (
            "<|system|>\n"
            "أنت خبير قانوني في صياغة العقود.\n"
            "<|user|>\n"
            f"المهمة: {inst}\n\n"
            f"النص الحالي:\n{bef}\n"
            "<|assistant|>\n"
            f"النص المعدل:\n{aft}"
        )
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

# =========================
# 7️⃣ Trainer
# =========================
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset=eval_dataset,   # ✅ add this
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "arabic_legal_outputs",
    ),
)

# =========================
# 8️⃣ Train
# =========================
torch.cuda.empty_cache()
trainer_stats = trainer.train()

# =========================
# 9️⃣ Save Model
# =========================
model.save_pretrained("arabic_legal_lora")
tokenizer.save_pretrained("arabic_legal_lora")

ImportError: Unsloth: Please install unsloth_zoo via `pip install unsloth_zoo` then retry!

In [ ]:
import time
import numpy as np
from tqdm import tqdm

def evaluate_model(model, tokenizer, eval_dataset, max_samples=50):

    model.eval()

    latencies = []
    correct = 0
    total = 0
    total_tokens = 0

    for example in tqdm(eval_dataset.select(range(min(max_samples, len(eval_dataset))))):

        prompt = example["text"]

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        start = time.time()

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.3,
                do_sample=False,
            )

        end = time.time()

        latency_ms = (end - start) * 1000
        latencies.append(latency_ms)

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Very simple accuracy proxy:
        # Check if expected answer exists in generated output
        expected_answer = example["text"].split("النص المعدل:")[-1].strip()

        if expected_answer in generated_text:
            correct += 1

        total += 1
        total_tokens += outputs.shape[1]

    accuracy = correct / total
    avg_latency = np.mean(latencies)

    # Example cost calculation
    # Suppose 3B 4bit on your GPU costs ~0.0008$ per 1k tokens
    cost_per_1k_tokens = 0.0008
    cost = (total_tokens / 1000) * cost_per_1k_tokens

    return {
        "accuracy": accuracy,
        "latency": avg_latency,
        "cost": cost,
    }

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Suc

In [ ]:

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from datasets import load_dataset
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# --- LoRA  ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj","gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

dataset = load_dataset("json", data_files="/content/drive/MyDrive/graduation project/input.json", split="train")

def format_prompts(examples):
    texts = []
    for inst, bef, aft in zip(examples["prompt"], examples["before"], examples["after"]):
        # (Prompt Template) for Qwen
        text = f"<|im_start|>system\nأنت خبير قانوني في صياغة العقود.<|im_end|>\n" \
               f"<|im_start|>user\nالمهمة: {inst}\n\nالنص الحالي:\n{bef}<|im_end|>\n" \
               f"<|im_start|>assistant\nالنص المعدل:\n{aft}<|im_end|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 20,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "legal_qwen_outputs",
    ),
)

# --- train---
torch.cuda.empty_cache()
trainer_stats = trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


FileNotFoundError: Unable to find '/content/drive/MyDrive/graduation project/input.json'

In [ ]:
save_directory = "/content/drive/MyDrive/graduation project/legal_llama32_lora"

model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(save_directory)


In [ ]:
!pip uninstall -y unsloth unsloth_zoo
!pip install -U pip

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U unsloth_zoo

!pip install xformers==0.0.34 trl==0.12.0 peft accelerate bitsandbytes


Found existing installation: unsloth 2026.2.1
Uninstalling unsloth-2026.2.1:
  Successfully uninstalled unsloth-2026.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-caw21o_o/unsloth_347120873e4a4355ab883cc06c63d732
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-caw21o_o/unsloth_347120873e4a4355ab883cc06c63d732
  Resolved https://github.com/unslothai/unsloth.git to commit 252502aa029260ed4d26b09ed8fe2f035fa34643
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.2.1-py3-none-any.whl size=442400 sha256=687890f77462cbcd1d0e7a366552cbb9b6d7ecf09fd92c

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U pip

!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121


Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.24.0+cu128
Uninstalling torchvision-0.24.0+cu128:
  Successfully uninstalled torchvision-0.24.0+cu128
Found existing installation: torchaudio 2.9.0+cu128
Uninstalling torchaudio-2.9.0+cu128:
  Successfully uninstalled torchaudio-2.9.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 21.3 MB/s  0:00:17
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 106.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 109.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 149.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 37.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 178.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 16.5 MB/s  0:00

In [ ]:
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -U unsloth_zoo
!pip install -U trl peft accelerate bitsandbytes datasets transformers


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2sg4pm4s/unsloth_7b5644a40d4b48748f6dd67bd10da0ea
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2sg4pm4s/unsloth_7b5644a40d4b48748f6dd67bd10da0ea
  Resolved https://github.com/unslothai/unsloth.git to commit 252502aa029260ed4d26b09ed8fe2f035fa34643
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.12.0
    Uninstalling trl-0.12.0:
      Successfully uninstalled trl-0.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 20.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 129.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 31.4 MB/s  

In [ ]:
!pip uninstall -y unsloth unsloth_zoo
!pip install -U unsloth==2025.12.8 unsloth_zoo==2025.12.8


Found existing installation: unsloth 2026.2.1
Uninstalling unsloth-2026.2.1:
  Successfully uninstalled unsloth-2026.2.1
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.5 MB/s  0:00:00
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl (915.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
print("torch:", torch.__version__)
from unsloth import FastLanguageModel
print("Unsloth OK")


torch: 2.10.0+cu128
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth OK


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/graduation project/legal_llama32_lora", # نفس المسار الذي حفظت فيه
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2025.12.8: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.12.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install rouge-score sentence-transformers


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import re
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

# موديل Embeddings (خفيف وسريع)
_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def _numbers_set(text: str):
    # يلمّ كل الأرقام كأعداد صحيحة (يفيد للعقود/الجداول)
    return set(int(x.replace(",", "")) for x in re.findall(r"\b\d{1,3}(?:,\d{3})*\b", text))

def text_similarity_rougeL(pred: str, gold: str) -> float:
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    f1 = scorer.score(gold, pred)["rougeL"].fmeasure
    return f1 * 100.0

def semantic_similarity(pred: str, gold: str) -> float:
    e1 = _embedder.encode(pred, convert_to_tensor=True)
    e2 = _embedder.encode(gold, convert_to_tensor=True)
    sim = util.cos_sim(e1, e2).item()
    return sim * 100.0

def clause_accuracy_generic(pred: str) -> tuple[float, list[str]]:
    """
    قواعد عامة جدًا (Generic) مش مرتبطة بمثال بعينه:
    - وجود عنوان المادة بعد التعديل
    - وجود جدول markdown (|---|)
    - وجود سطر الإجمالي
    - وجود سطر نسبة المشاركة (لو موجود عادة في صياغتك)
    """
    checks = []
    reasons = []

    checks.append(("has_article_title", "المادة" in pred and "بعد التعديل" in pred))
    checks.append(("has_table_sep", "|---" in pred))
    checks.append(("has_total", ("الاجمالي" in pred) or ("الإجمالي" in pred)))
    checks.append(("has_egypt_100", "نسبة المشاركة المصرية 100%" in pred or "المشاركة المصرية 100%" in pred))

    ok = sum(int(v) for _, v in checks)
    total = len(checks)

    for name, v in checks:
        if not v:
            reasons.append(f"ClauseAccuracy fail: {name}")

    return (ok / total) * 100.0, reasons

def change_accuracy_against_gold(pred: str, gold: str) -> tuple[float, list[str]]:
    """
    تعميم عملي: طالما عندك gold، نقيس تنفيذ التغييرات عبر:
    - تشابه الأرقام المهمة (Intersection/Recall)
    - وجود أسماء الأطراف (تقريبي) عبر كلمات عربية طويلة (اختياري)
    """
    reasons = []

    gold_nums = _numbers_set(gold)
    pred_nums = _numbers_set(pred)

    if not gold_nums:
        num_score = 100.0
    else:
        # Recall على أرقام الـ gold
        recall = len(pred_nums & gold_nums) / len(gold_nums)
        num_score = recall * 100.0
        if recall < 1.0:
            missing = sorted(list(gold_nums - pred_nums))[:10]
            reasons.append(f"ChangeAccuracy: missing some gold numbers (e.g. {missing})")

    return num_score, reasons

def evaluate_sample(pred: str, gold: str) -> dict:
    ts = text_similarity_rougeL(pred, gold)
    ss = semantic_similarity(pred, gold)

    ca, ca_reasons = clause_accuracy_generic(pred)
    cha, cha_reasons = change_accuracy_against_gold(pred, gold)

    legal = 0.5 * ca + 0.5 * cha
    final = 0.6 * legal + 0.2 * ts + 0.2 * ss

    return {
        "TextSimilarity_ROUGE-L": round(ts, 2),
        "SemanticSimilarity": round(ss, 2),
        "ClauseAccuracy": round(ca, 2),
        "ChangeAccuracy": round(cha, 2),
        "LegalAccuracy": round(legal, 2),
        "FinalScore": round(final, 2),
        "Notes": ca_reasons + cha_reasons,
    }


In [ ]:
gold = """المادة السادسة بعد التعديل:
حدد رأس مال الشركة بمبلغ 50000 (خمسون ألف جنيه مصري)، موزع إلى عدد 50000 حصة قيمة كل منها (1) جنيه مصري (واحد جنيه مصري) وجميعها حصص نقدية وقد تم توزيع هذه الحصص بين الشركاء على الوجه الآتي:
| م | اسم صاحب الحصة | جنسيته | عدد الحصص النقدية | القيمة بالجنيه المصري | العملة التي تم الوفاء بها |
|---|-------------------|---------|-------------------|------------------------|---------------------------|
| 1 | محمد عبدالله إبراهيم دسوقى | - | 12500 | 12500 | - |
| 2 | عمرو احمد إبراهيم احمد على | مصري | 25000 | 25000 | - |
| 3 | عمر عبدالله احمد محمد | مصري | 12500 | 12500 | - |
| | الاجمالي | - | 50000 | 50000 | - |
وتبلغ نسبة المشاركة المصرية 100%"""


In [ ]:
instruction = "تعديل هيكل الملكية وتوزيع الحصص بين الشركاء مع ثبات رأس المال عند 50000 جنيه؛ حيث تم خروج الشريك (حسام مجدى أبو السعود محمد) ودخول شريك جديد (محمد عبدالله إبراهيم دسوقى) بحصة 12500، مع زيادة حصة الشريك (عمرو احمد إبراهيم احمد على) من 16500 إلى 25000، وتخفيض حصة الشريك (عمر عبدالله احمد محمد) من 16500 إلى 12500."

messages = [
  {"role":"system","content":"أنت خبير قانوني في صياغة العقود."},
  {"role":"user","content": f"""
اكتب فقط "المادة السادسة بعد التعديل" بنفس تنسيق الجدول.

شروط إلزامية:
- ممنوع ظهور الاسم التالي نهائياً في أي سطر: (حسام مجدى أبو السعود محمد).
- جدول الشركاء بعد التعديل يحتوي 3 صفوف فقط (بدون حسام):
  1) محمد عبدالله إبراهيم دسوقى = 12500
  2) عمرو احمد إبراهيم احمد على = 25000
  3) عمر عبدالله احمد محمد = 12500
- الإجمالي = 50000 حصة.
- إذا ظهر اسم حسام في الرد، أعد كتابة الجدول من البداية بدون حسام.

النص الحالي:
المادة السادسة قبل التعديل:
حدد رأس مال الشركة بمبلغ 50000 (خمسون ألف جنيه مصري)، موزع إلى عدد 50000 حصة قيمة كل منها (1) جنيه مصري (واحد جنيه مصري) وجميعها حصص نقدية وقد تم توزيع هذه الحصص بين الشركاء على الوجه الآتي:
| م | اسم صاحب الحصة | جنسيته | عدد الحصص النقدية | القيمة بالجنيه المصري | العملة التي تم الوفاء بها |
|---|-------------------|---------|-------------------|------------------------|---------------------------|
| 1 | حسام مجدى أبو السعود محمد | مصري | 17000 | 17000 | جنيه مصري |
| 2 | عمر عبدالله احمد محمد | مصري | 16500 | 16500 | جنيه مصري |
| 3 | عمرو احمد إبراهيم احمد على | مصري | 16500 | 16500 | جنيه مصري |
| | الاجمالي | - | 50000 | 50000 | جنيه مصري |
وتبلغ نسبة المشاركة المصرية 100%
"""
  },
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.0,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)


# اطبع الرد فقط بدون system/user
gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(gen_ids, skip_special_tokens=True).strip())



gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
pred = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
result = evaluate_sample(pred, gold)
print(result)


النص المعدل:
المادة السادسة بعد التعديل:
حدد رأس مال الشركة بمبلغ 50000 (خمسون ألف جنيه مصري)، موزع إلى عدد 50000 حصة قيمة كل منها (1) جنيه مصري (واحد جنيه مصري) وجميعها حصص نقدية وقد تم توزيع هذه الحصص بين الشركاء على الوجه الآتي:
| م | اسم صاحب الحصة | جنسيته | عدد الحصص النقدية | القيمة بالجنيه المصري | العملة التي تم الوفاء بها |
|---|-------------------|---------|-------------------|------------------------|---------------------------|
| 1 | محمد عبدالله إبراهيم دسوقى | مصري | 12500 | 12500 | جنيه مصري |
| 2 | عمرو احمد إبراهيم احمد على | مصري | 25000 | 25000 | جنيه مصري |
| 3 | عمر عبدالله احمد محمد | مصري | 12500 | 12500 | جنيه مصري |
| | الاجمالي | - | 50000 | 50000 | جنيه مصري |
وتبلغ نسبة المشاركة المصرية 100%
{'TextSimilarity_ROUGE-L': 100.0, 'SemanticSimilarity': 97.84, 'ClauseAccuracy': 100.0, 'ChangeAccuracy': 100.0, 'LegalAccuracy': 100.0, 'FinalScore': 99.57, 'Notes': []}
